In [ ]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from tqdm import tqdm

# ─────────────────────────────────────────────
# CONFIG — update these paths
# ─────────────────────────────────────────────
DATA_DIR = Path("../data/m5")
SUBMISSIONS_DIR = DATA_DIR / "uncertainty_submissions"
SALES_PATH = DATA_DIR / "sales_train_evaluation.csv"
CALENDAR_PATH = DATA_DIR / "calendar.csv"

# The 9 quantile levels used in the M5 Uncertainty competition
QUANTILE_LEVELS = [0.005, 0.025, 0.165, 0.250, 0.500, 0.750, 0.835, 0.975, 0.995]

# We want forecasts for day 7 of the 28-day horizon (0-indexed → index 6)
FORECAST_DAY = 7  # "day 7" as described in the paper (1-indexed)

# Paper threshold: keep forecasters with ≥ 10,000 product-store pairs on day 7
MIN_PAIRS = 10_000


# ─────────────────────────────────────────────
# STEP 1: LOAD GROUND-TRUTH SALES
# ─────────────────────────────────────────────

def load_ground_truth(sales_path: Path, calendar_path: Path) -> pd.DataFrame:
    """
    Load the actual sales data and return a DataFrame with binary labels:
    y=1 if at least 1 unit was sold, y=0 otherwise.

    The M5 evaluation uses the last 28 days of the evaluation file as the
    forecast horizon. We focus on day 7 of that horizon.
    """
    print("Loading ground-truth sales data...")
    sales = pd.read_csv(sales_path)
    calendar = pd.read_csv(calendar_path)

    # The last 28 columns of sales_train_evaluation are the test horizon (d_1914 to d_1941)
    # Identify day-columns
    day_cols = [c for c in sales.columns if re.match(r"^d_\d+$", c)]

    # The evaluation horizon is the last 28 days
    horizon_cols = day_cols[-28:]

    # Day 7 of the horizon (1-indexed) → index 6
    target_col = horizon_cols[FORECAST_DAY - 1]
    print(f"  Using sales column: {target_col} (day {FORECAST_DAY} of 28-day horizon)")

    # Build product-store identifier
    sales["product_store"] = sales["item_id"] + "__" + sales["store_id"]

    # Create binary label: 1 if at least one unit sold
    labels = sales[["product_store", "item_id", "store_id", target_col]].copy()
    labels.rename(columns={target_col: "actual_sales"}, inplace=True)
    labels["y"] = (labels["actual_sales"] >= 1).astype(int)

    print(f"  Total product-store pairs: {len(labels):,}")
    print(f"  Positive rate (at least 1 sale): {labels['y'].mean():.3f}")
    return labels

In [ ]:

# ─────────────────────────────────────────────
# STEP 2: LOAD & PARSE A SINGLE SUBMISSION
# ─────────────────────────────────────────────

def parse_submission(filepath: Path) -> pd.DataFrame:
    """
    Parse one M5 Uncertainty submission CSV.

    Each submission has rows like:
        id, F1, F2, ..., F28
    where the id encodes the series and quantile level, e.g.:
        FOODS_3_090_CA_1_validation_0.005
                                     ^^^^^ quantile
    Returns a DataFrame with columns:
        product_store, quantile, p_hat
    where p_hat is the quantile forecast for FORECAST_DAY.
    """
    df = pd.read_csv(filepath)

    # Parse the id column to extract series and quantile
    # Format: <item_id>_<store_id>_validation_<quantile>  (level 12 = item x store)
    # or higher aggregation levels — we only want level 12
    df["quantile"] = df["id"].str.extract(r"_(\d+\.\d+)$").astype(float)
    df["base_id"] = df["id"].str.replace(r"_\d+\.\d+$", "", regex=True)

    # Keep only item x store level: id has format DEPT_NUM_ITEM_STATE_STORE_validation_q
    # Level 12 ids look like: FOODS_3_090_CA_1_validation_0.005
    # They have exactly 5 underscore-delimited parts before "_validation"
    mask = df["base_id"].str.match(
        r"^[A-Z]+_\d+_\d+_[A-Z]{2}_\d+_(?:validation|evaluation)$"
    )
    df = df[mask].copy()

    if df.empty:
        return pd.DataFrame()

    # The forecast columns are F1..F28; pick FORECAST_DAY
    forecast_col = f"F{FORECAST_DAY}"
    if forecast_col not in df.columns:
        raise ValueError(f"Column {forecast_col} not found in {filepath.name}")

    df = df[["base_id", "quantile", forecast_col]].copy()
    df.rename(columns={forecast_col: "q_forecast"}, inplace=True)

    # Reconstruct product_store from base_id
    # base_id = FOODS_3_090_CA_1_validation → item_id = FOODS_3_090, store_id = CA_1
    def extract_product_store(base_id: str) -> str:
        parts = base_id.split("_")
        # Remove trailing "validation" or "evaluation"
        parts = [p for p in parts if p not in ("validation", "evaluation")]
        # item_id = first 3 parts, store_id = last 2 parts
        item_id = "_".join(parts[:3])
        store_id = "_".join(parts[3:5])
        return f"{item_id}__{store_id}"

    df["product_store"] = df["base_id"].apply(extract_product_store)
    return df[["product_store", "quantile", "q_forecast"]]


# ─────────────────────────────────────────────
# STEP 3: CDF INTERPOLATION → P(sales ≥ 1)
# ─────────────────────────────────────────────

def quantiles_to_prob_at_least_one(group: pd.DataFrame) -> float:
    """
    Given a group of (quantile, q_forecast) rows for ONE product-store pair,
    linearly interpolate the CDF and return P(sales >= 1) = 1 - CDF(0).

    The paper uses linear interpolation to convert quantile forecasts into
    estimates of the full CDF, then sets p_hat = 1 - CDF(0).
    """
    q_levels = group["quantile"].values
    q_values = group["q_forecast"].values

    # Sort by quantile level
    order = np.argsort(q_levels)
    q_levels = q_levels[order]
    q_values = q_values[order]

    # Enforce monotonicity of quantile values (CDF must be non-decreasing)
    q_values = np.maximum.accumulate(q_values)

    # Edge case: all forecasts are 0 → P(sales >= 1) = 0
    if q_values[-1] == 0:
        return 0.0

    # Add boundary anchors for extrapolation:
    # At sales = -∞ (use -1), CDF ≈ 0; at sales = q_values[-1] + buffer, CDF ≈ 1
    sales_vals = np.concatenate([[-1.0], q_values, [q_values[-1] + 100]])
    cdf_vals = np.concatenate([[0.0], q_levels, [1.0]])

    # Linear interpolation of CDF at sales = 0
    interp_fn = interp1d(sales_vals, cdf_vals, kind="linear",
                         bounds_error=False,
                         fill_value=(0.0, 1.0))
    cdf_at_zero = float(interp_fn(0.0))

    # P(sales >= 1) = 1 - P(sales < 1) ≈ 1 - CDF(0)
    # (Since sales are discrete integers, P(sales >= 1) = 1 - P(sales = 0) = 1 - CDF(0))
    p_hat = 1.0 - cdf_at_zero
    return float(np.clip(p_hat, 0.0, 1.0))


# ─────────────────────────────────────────────
# STEP 4: PROCESS ALL SUBMISSIONS
# ─────────────────────────────────────────────

def process_all_submissions(submissions_dir: Path,
                            labels: pd.DataFrame) -> pd.DataFrame:
    """
    Load all top-50 submissions, compute p_hat for each forecaster,
    filter to ≥ MIN_PAIRS product-store pairs (as in the paper),
    and merge with ground-truth labels.

    Returns a DataFrame with columns:
        product_store, y, forecaster, p_hat
    """
    submission_files = sorted(submissions_dir.glob("*.csv"))
    if not submission_files:
        raise FileNotFoundError(
            f"No CSV files found in {submissions_dir}. "
            "Did you unzip the submissions?"
        )

    print(f"\nFound {len(submission_files)} submission files.")

    valid_pairs = set(labels["product_store"])
    all_results = []

    for fpath in tqdm(submission_files, desc="Processing submissions"):
        forecaster_name = fpath.stem  # e.g., "0001 YJ_STU"
        try:
            df = parse_submission(fpath)
        except Exception as e:
            print(f"  ⚠ Skipped {fpath.name}: {e}")
            continue

        if df.empty:
            print(f"  ⚠ No level-12 rows found in {fpath.name}")
            continue

        # Keep only product-store pairs that exist in our ground truth
        df = df[df["product_store"].isin(valid_pairs)]

        # Count distinct product-store pairs with all 9 quantiles present
        pair_counts = df.groupby("product_store")["quantile"].count()
        complete_pairs = pair_counts[pair_counts == len(QUANTILE_LEVELS)].index

        if len(complete_pairs) < MIN_PAIRS:
            print(f"  ✗ {forecaster_name}: only {len(complete_pairs):,} complete pairs "
                  f"(need ≥ {MIN_PAIRS:,}) — excluded")
            continue

        df = df[df["product_store"].isin(complete_pairs)]

        # Compute p_hat = P(sales >= 1) via CDF interpolation
        p_hats = (
            df.groupby("product_store")
              .apply(quantiles_to_prob_at_least_one)
              .reset_index()
        )
        p_hats.columns = ["product_store", "p_hat"]
        p_hats["forecaster"] = forecaster_name

        all_results.append(p_hats)
        print(f"  ✓ {forecaster_name}: {len(p_hats):,} pairs processed")

    print(f"\n{len(all_results)} forecasters passed the ≥{MIN_PAIRS:,} pairs filter.")

    if not all_results:
        raise RuntimeError("No valid submissions found after filtering.")

    # Combine all forecasters and merge with ground-truth labels
    combined = pd.concat(all_results, ignore_index=True)
    result = combined.merge(
        labels[["product_store", "y", "item_id", "store_id"]],
        on="product_store",
        how="left"
    )

    return result


# ─────────────────────────────────────────────
# STEP 5: ADDITIONAL PREPROCESSING
# ─────────────────────────────────────────────

def additional_preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    """
    Additional preprocessing steps useful for algorithm implementation:
    1. Clip p_hat to avoid exact 0/1 (avoids log(0) in loss functions)
    2. Add a unique index per (product_store, forecaster)
    3. Sort for reproducibility
    """
    EPSILON = 1e-6
    df["p_hat"] = df["p_hat"].clip(EPSILON, 1 - EPSILON)
    df = df.sort_values(["product_store", "forecaster"]).reset_index(drop=True)
    return df


# ─────────────────────────────────────────────
# STEP 6: SAVE OUTPUTS
# ─────────────────────────────────────────────

def save_outputs(df: pd.DataFrame, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)

    # Full long-format dataset (one row per product-store x forecaster)
    long_path = out_dir / "m5_preprocessed_long.csv"
    df.to_csv(long_path, index=False)
    print(f"\nSaved long-format data: {long_path}")

    # Wide-format: rows = product-store pairs, columns = one per forecaster
    wide = df.pivot(index="product_store", columns="forecaster", values="p_hat")
    wide = wide.join(
        df.drop_duplicates("product_store").set_index("product_store")[["y"]]
    )
    wide_path = out_dir / "m5_preprocessed_wide.csv"
    wide.to_csv(wide_path)
    print(f"Saved wide-format data:  {wide_path}")

    # Summary statistics
    print("\n── Summary ──────────────────────────────")
    print(f"  Unique product-store pairs: {df['product_store'].nunique():,}")
    print(f"  Forecasters retained:       {df['forecaster'].nunique():,}")
    print(f"  Overall positive rate (y=1): {df.drop_duplicates('product_store')['y'].mean():.3f}")
    print(f"  p_hat range:  [{df['p_hat'].min():.4f}, {df['p_hat'].max():.4f}]")
    print(f"  p_hat mean:   {df['p_hat'].mean():.4f}")
    print("─────────────────────────────────────────")

    return wide


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    # 1. Load ground truth
    labels = load_ground_truth(SALES_PATH, CALENDAR_PATH)

    # 2. Process all submissions → compute p_hat for each forecaster
    df = process_all_submissions(SUBMISSIONS_DIR, labels)

    # 3. Additional preprocessing
    df = additional_preprocessing(df)

    # 4. Save
    out_dir = DATA_DIR / "processed"
    wide = save_outputs(df, out_dir)

    print("\nDone! Files ready for your algorithm:")
    print("  • m5_preprocessed_long.csv — one row per (product_store, forecaster)")
    print("  • m5_preprocessed_wide.csv — one row per product_store, forecasters as columns")
    print("\nKey columns:")
    print("  y      — binary label (1 = at least 1 unit sold)")
    print("  p_hat  — forecaster's estimated P(sales ≥ 1) via CDF interpolation")
    return df, wide


if __name__ == "__main__":
    df, wide = main()